[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niccoloridi/International-Law-and-AI/blob/main/IntlLaw_AI_Lab08_Information_Warfare.ipynb)

In [ ]:
#@title Lab 08 — Information Warfare
print("""
 █████             █████    █████                                                     █████████   █████
░░███             ░░███    ░░███                                         ███         ███░░░░░███ ░░███
 ░███  ████████   ███████   ░███         ██████   █████ ███ █████       ░███        ░███    ░███  ░███
 ░███ ░░███░░███ ░░░███░    ░███        ░░░░░███ ░░███ ░███░░███     ███████████    ░███████████  ░███
 ░███  ░███ ░███   ░███     ░███         ███████  ░███ ░███ ░███    ░░░░░███░░░     ░███░░░░░███  ░███
 ░███  ░███ ░███   ░███ ███ ░███      █ ███░░███  ░░███████████         ░███        ░███    ░███  ░███
 █████ ████ █████  ░░█████  ███████████░░████████  ░░████░████          ░░░         █████   █████ █████
░░░░░ ░░░░ ░░░░░    ░░░░░  ░░░░░░░░░░░  ░░░░░░░░    ░░░░ ░░░░                      ░░░░░   ░░░░░ ░░░░░


                                                                                                       

   Lab 08 // Information Warfare
   Melbourne Law Masters 2026
""")

*Brought to you by **Dr Niccolò Ridi**, Purveyor of Fine Scripts* – [KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)

# Lab 08: Information Warfare – Generating and Analysing Propaganda

**Session 8 of International Law and AI – Melbourne Law Masters 2026**

## Overview

This lab does three things. First, we build two propaganda generators – one that fabricates, one that distorts – and watch what a real frontier model produces when asked to make each. Second, we probe the model's refusal surface: what framings bypass guardrails, what framings do not. Third, we turn the same technology around and try to detect what we just generated, including whether it plausibly crosses the coercion threshold the ICJ set out in *Nicaragua*.

The doctrinal anchor is Marko Milanovic's distinction between **false news** (fabricated factual claims) and **distorted news** (factually accurate components presented misleadingly). The legal consequence: false news plus coercion can breach the non-intervention principle; distorted news almost never does. We will end the lab with a hands-on thought experiment: given the 1936 Broadcasting Convention and the 1953 Correction Convention presumed *identifiable state broadcasters*, what would those conventions have to look like today?

## Where this lab sits on the AI map

Three broad paradigms, one rough map. This course uses all three.

| Paradigm | What it does | How | Example tools | This lab |
| --- | --- | --- | --- | --- |
| Symbolic | Follows explicit rules | Humans write logic; machine executes | IF-THEN rules, expert systems, treaty-as-code | Labs 01 (P1), 07 (P1) |
| **Subsymbolic (pattern recognition)** | Learns to classify, detect, or measure similarity | Neural network trained on labelled data; no explicit rules | CNNs (MobileNet, YOLO), BERT embeddings, sentence-transformers | Labs 01 (P2), 04, 06, 07 (P3), 08 (P4), 09 (P3–4), 10 (P6) |
| **Generative** | Produces new text, image, or action | Large model predicts next token from a probability distribution | GPT, Gemini, Claude; agentic systems layer tools on top | Labs 02, 05, 08 (P2–3), 09 (P1–2), 10 (P3–5) |

Generative models are technically subsymbolic too – they are neural networks under the hood. They are separated out because their behaviour (producing new content rather than classifying existing content) poses distinctive legal problems: authorship, attribution, hallucination, autonomy.

**This lab sits in:** **Generative** (Parts Two–Three) and **Subsymbolic** (Part Four).

*This lab sits on both sides of the arms race – generating synthetic media and detecting it.*

In [ ]:
#@title Paper notes setup
#@markdown This lab will append your reflections to a markdown file you can download at the end of the session and use as material for your 5,000-word paper.

from pathlib import Path
from datetime import datetime

NOTES_PATH = Path("/content/paper_notes_lab_08.md")
if not NOTES_PATH.exists():
    NOTES_PATH.write_text(
        f"# Lab 08 – paper notes\n\nStarted: {datetime.now().isoformat(timespec='minutes')}\n\n"
    )

def _append_to_notes(heading, body):
    with NOTES_PATH.open("a") as f:
        f.write(f"\n## {heading}\n\n{body}\n")

print(f"Notes will accumulate in: {NOTES_PATH}")
print("At the end of the lab, open the Files panel (folder icon, left) to download.")

# Part One // The Information Environment

On 30 October 1938, Orson Welles's *War of the Worlds* radio broadcast convinced some listeners – how many remains contested – that Martians were landing in New Jersey. The point is not the panic. The point is that it happened with one voice, on one frequency, on one evening. The machinery of mass belief was already a machinery of one-to-many communication, and state law had not yet noticed.

We are in a different machinery now. What a lone operator with a Gemini API key can produce in an afternoon is what state broadcasters used to need an apparatus for. International law's answer to that machinery is mostly silence.

In this Part: install the tools, set up the API, meet the Milanovic distinction, and declare the three legal-classification functions we will implement over the rest of the lab.

In [ ]:
#@title Install dependencies
!pip install -q google-genai pandas matplotlib seaborn
print("✓ Installed google-genai, pandas, matplotlib, seaborn.")

In [ ]:
#@title API key setup

from google import genai
from google.genai import types

# Try Colab Secrets first; fall back to getpass for local use.
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY not set in Colab Secrets.")
    print("✓ API key loaded from Colab Secrets.")
except (ImportError, Exception) as e:
    from getpass import getpass
    api_key = getpass("Enter GEMINI_API_KEY: ")
    print("✓ API key provided via getpass.")

client = genai.Client(api_key=api_key)
print("Using model: gemini-3-flash-preview")

## The Milanovic Distinction

Marko Milanovic's *"Revisiting Coercion as an Element of Prohibited Intervention in International Law"* (AJIL 118:2, 2024) draws a distinction that most of the disinformation literature does not. **False news** is a fabricated factual statement – a claim that something happened when it did not, or that a person said something they did not. **Distorted news** uses factually accurate components (a real quote, a real image, a real event) but presents them in a manner engineered to mislead: selective omission, misleading juxtaposition, framing that invites an inference the facts do not support.

The legal bite of the distinction lives in Article 2(1) of the UN Charter and the prohibition of intervention in the internal affairs of other states. Per *Nicaragua* (ICJ 1986, paras 205 and 241), prohibited intervention requires **coercion** – it is not enough for conduct to be harmful, unwelcome, or malicious. False news *can* cross that threshold, because it can shape political outcomes (elections, military decisions, civil order) in ways the target state has no ability to counter. Distorted news almost never does, because the components are truthful and the mechanism operates through the target population's own interpretation. The paradigmatic hybrid case is the **Lisa Case (January 2016)**: Russian state media reported that a 13-year-old Russian-German girl in Berlin had been raped by refugees. The rape never happened (false). The broader story tapped into genuine anxieties about migrant integration (distorted context). Lavrov escalated the matter at foreign-minister level. Protests followed. Whether coercion was established remains contested.

In [ ]:
#@title Classification function stubs

# These are the three legal categories Part Four will implement. Declaring them
# here, empty, makes the categories *nameable*. You can point at the stubs and
# say: these are the things we want to be able to compute about a piece of text.

def is_false_news(claim: str) -> dict:
    """Classify a claim as likely_false / likely_true / uncertain.

    Stub: implemented in Part Four.
    """
    raise NotImplementedError("Implemented in Part Four")


def is_distorted_news(claim: str, context: str) -> dict:
    """Classify whether a claim distorts the supplied factual context.

    Stub: implemented in Part Four.
    """
    raise NotImplementedError("Implemented in Part Four")


def crosses_coercion_threshold(claim: str, target_state: str) -> dict:
    """Classify whether the claim, if disseminated against the target state,
    plausibly meets the Nicaragua coercion threshold.

    Stub: implemented in Part Four.
    """
    raise NotImplementedError("Implemented in Part Four")


classification_stubs_loaded = True
print("✓ is_false_news / is_distorted_news / crosses_coercion_threshold – stubs declared.")
print("  Bodies land in Part Four.")

### So what just happened?

You have named three things as *functions*: false news, distorted news, coercion. Declaring them this way has a pedagogical edge. A function has a signature, a return type, and a body. The Milanovic distinction says the first two are different legal objects, and the coercion test says only some of them trigger a prohibition. None of that is visible to you until you try to *compute* it – and computing it requires a concrete input, a concrete output, and a willingness to say where the bodies break.

Part Four will give each function a body. Between now and then, we will generate the inputs those functions will operate on. That is Part Two.